In [1]:
!pip install -q datasets librosa tensorflow scikit-learn matplotlib

In [2]:
import numpy as np
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt

from datasets import load_dataset
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
ds = load_dataset(
    "SpeechAntiSpoofingBenchmarks/DFADD",
    split="test"
)

print(ds)
print("Dataset columns:", ds.column_names)

README.md:   0%|          | 0.00/3.70k [00:00<?, ?B/s]

data/test-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  302MB            

data/test-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  239MB            

data/test-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/3755 [00:00<?, ? examples/s]

Dataset({
    features: ['path', 'audio', 'label', 'notes'],
    num_rows: 3755
})
Dataset columns: ['path', 'audio', 'label', 'notes']


In [4]:
X = []
y = []

for i, row in enumerate(ds):
    audio = row["audio"]
    signal = np.asarray(audio["array"], dtype=np.float32)
    sr = audio["sampling_rate"]

    # Generate mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=128
    )

    # Convert power to decibels
    mel = librosa.power_to_db(mel, ref=np.max)

    # Normalize to 0–1
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)

    # Resize to 128 x 128
    mel = tf.image.resize(
        mel[..., np.newaxis],
        (128, 128)
    ).numpy()

    # Convert grayscale to 3 channels
    mel = np.repeat(mel, 3, axis=-1) * 255.0

    X.append(mel)
    y.append(row["label"])

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1} samples")

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Labels:", np.unique(y))

Processed 100 samples
Processed 200 samples
Processed 300 samples
Processed 400 samples
Processed 500 samples
Processed 600 samples
Processed 700 samples
Processed 800 samples
Processed 900 samples
Processed 1000 samples
Processed 1100 samples
Processed 1200 samples
Processed 1300 samples
Processed 1400 samples
Processed 1500 samples
Processed 1600 samples
Processed 1700 samples
Processed 1800 samples
Processed 1900 samples
Processed 2000 samples
Processed 2100 samples
Processed 2200 samples
Processed 2300 samples
Processed 2400 samples
Processed 2500 samples
Processed 2600 samples
Processed 2700 samples
Processed 2800 samples
Processed 2900 samples
Processed 3000 samples
Processed 3100 samples
Processed 3200 samples
Processed 3300 samples
Processed 3400 samples
Processed 3500 samples
Processed 3600 samples
Processed 3700 samples
X shape: (3755, 128, 128, 3)
y shape: (3755,)
Labels: [0 1]


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 3004
Testing samples: 751


In [5]:
# Load pretrained ResNet50
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze pretrained layers for initial training
base_model.trainable = False

# Add classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

# Binary output: real or fake
output = Dense(1, activation="sigmoid")(x)

# Create complete model
model = Model(
    inputs=base_model.input,
    outputs=output,
    name="VoiceGuard_ResNet50"
)

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "VoiceGuard_ResNet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 134, 134,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 64, 64,    │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 64, 64,    │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 64, 64,    │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 66, 66,    │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 32, 32,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 32, 32,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 32, 32,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 32, 32,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 32, 32,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 32, 32,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 32, 32,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 32, 32,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 32, 32,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 32, 32,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 32, 32,    │      1,024 │ conv2_block1_3_c

 Total params: 24,112,513 (91.98 MB)

 Trainable params: 524,801 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [7]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

Epoch 1/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 177s 1s/step - accuracy: 0.8735 - loss: 0.3144 - val_accuracy: 0.9584 - val_loss: 0.1558 - learning_rate: 1.0000e-04
Epoch 2/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 207s 1s/step - accuracy: 0.9405 - loss: 0.1592 - val_accuracy: 0.9667 - val_loss: 0.1150 - learning_rate: 1.0000e-04
Epoch 3/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 201s 1s/step - accuracy: 0.9571 - loss: 0.1219 - val_accuracy: 0.9684 - val_loss: 0.0963 - learning_rate: 1.0000e-04
Epoch 4/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 202s 1s/step - accuracy: 0.9655 - loss: 0.0992 - val_accuracy: 0.9750 - val_loss: 0.0829 - learning_rate: 1.0000e-04
Epoch 5/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 174s 1s/step - accuracy: 0.9734 - loss: 0.0868 - val_accuracy: 0.9767 - val_loss: 0.0729 - learning_rate: 1.0000e-04
Epoch 6/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 174s 1s/step - accuracy: 0.9775 - loss: 0.0701 - val_accuracy: 0.9784 - val_loss: 0.0621 - learning_rate: 1.0000e-04
Epoch 7/10
151/151 ━━━━━━━━━━━━━━━━━━━━ 202s 1s/step - acc

In [8]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Generate predictions
y_prob = model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int).ravel()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

24/24 ━━━━━━━━━━━━━━━━━━━━ 68s 3s/step - accuracy: 0.9854 - loss: 0.0509
Test Loss: 0.0509
Test Accuracy: 98.54%
24/24 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96       151
           1       0.99      0.99      0.99       600

    accuracy                           0.99       751
   macro avg       0.97      0.98      0.98       751
weighted avg       0.99      0.99      0.99       751


Confusion Matrix:
[[148   3]
 [  8 592]]
